# 00 · Start here

Run the cell below **once** to confirm you're ready. It checks the few things the workshop needs and
tells you exactly what to fix if something's missing — *before* we're mid-demo. The required checks need
**no Docker and no API key**; the optional ones are only for the live parts.

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(ROOT / ".env")
OK, OPT, BAD = "\u2713", "\u25cb", "\u2717"
problems = []

def need(label, cond, fix):
    print(f"  {OK if cond else BAD} {label}")
    if not cond:
        print(f"      \u2192 {fix}")
        problems.append(label)

def opt(label, cond, note):
    print(f"  {OK if cond else OPT} {label}" + ("" if cond else f"   ({note})"))

print("REQUIRED — the workshop needs these:")
try:
    import context_workshop  # noqa: F401
    pkg = True
except Exception:
    pkg = False
need("context_workshop importable (uv sync ran)", pkg, "run:  uv sync")

DOC = ROOT / "data/cache/cargo-target/doc"
LIB = ROOT / "data/raw_repos/qdrant/lib"
CRATES = ["posting_list", "sparse", "gridstore", "quantization"]
need("rustdoc JSON for 4 crates (notebook 01, Rung 1)",
     all((DOC / f"{c}.json").exists() for c in CRATES), "run:  scripts/setup_data.sh")
need("raw source for those crates (notebook 01, Rung 0)",
     all((LIB / c).exists() for c in CRATES), "run:  scripts/setup_data.sh")
need("committed eval cache (notebook 02 renders with no key)",
     (ROOT / "data/eval/graph_vs_grep_cache.json").exists() and (ROOT / "data/eval/gold.json").exists(),
     "the repo ships this \u2014 re-clone if it's missing")

print("\nOPTIONAL — only for the live parts:")
neo4j_up = False
try:
    from context_workshop.graph import connect
    d = connect(os.getenv("NEO4J_URI", "neo4j://localhost:7687"), "neo4j", os.getenv("NEO4J_PASSWORD", "workshop123"))
    with d.session() as s:
        s.run("RETURN 1").single()
    d.close()
    neo4j_up = True
except Exception:
    pass
opt("Neo4j reachable (notebook 01, Level 3)", neo4j_up, "start it with:  docker compose up -d")
opt("OPENROUTER_API_KEY set (live agent + fenic cells)", bool(os.getenv("OPENROUTER_API_KEY")),
    "add it to .env \u2014 optional; everything else runs without it")

print("\n" + "=" * 56)
if problems:
    print(f"{BAD} Not ready yet \u2014 fix the {len(problems)} item(s) above, then re-run this cell.")
else:
    print(f"{OK} You're ready.")
    print("   \u2022 notebook 02 \u2014 the eval result (no Docker, no key). Start here.")
    print("   \u2022 notebook 01 \u2014 build the graph and query it (Level 3 needs Neo4j).")
    print("   \u2022 notebooks 03 / 04 \u2014 Python + fenic codas (optional).")

REQUIRED — the workshop needs these:
  ✓ context_workshop importable (uv sync ran)
  ✓ rustdoc JSON for 4 crates (notebook 01, Rung 1)
  ✓ raw source for those crates (notebook 01, Rung 0)
  ✓ committed eval cache (notebook 02 renders with no key)

OPTIONAL — only for the live parts:
  ✓ Neo4j reachable (notebook 01, Level 3)
  ✓ OPENROUTER_API_KEY set (live agent + fenic cells)

✓ You're ready.
   • notebook 02 — the eval result (no Docker, no key). Start here.
   • notebook 01 — build the graph and query it (Level 3 needs Neo4j).
   • notebooks 03 / 04 — Python + fenic codas (optional).
